In [1]:
import threading
from bleak import BleakClient
import asyncio
import time

class BleakClientThread:
    """
    A class to manage asynchronous BLE operations using Bleak and threading.
    see ref at https://github.com/hbldh/bleak

    Attributes:
        address (str): The BLE device address.
        loop (asyncio.new_event_loop): The event loop for asynchronous operations.
        client (BleakClient): The Bleak BLE client.
        thread (threading.Thread): The thread running the event loop.
        connected (bool): Connection status of the BLE client.

    Methods:
        run_loop(loop): Runs the given event loop.
        connect(): Asynchronously connects to the BLE device.
        disconnect(): Asynchronously disconnects from the BLE device.
        read_characteristic(char_uuid): Reads data from a specified characteristic.
        write_characteristic(char_uuid, data): Writes data to a specified characteristic.
        start(): Starts the thread and the event loop.
        stop(): Stops the event loop and joins the thread.
        perform_action(coro): Schedules a coroutine to be run in the event loop thread.
        prot_open(): Placeholder for opening protocol.
        prot_close(): Placeholder for closing protocol.
        prot_stop(): Placeholder for stopping protocol.
    """
    def __init__(self, address):
        """
        Initializes the BleakClientThread with a given BLE device address.

        Parameters:
            address (str): The BLE device address.
        """

        self.address = address
        self.loop = asyncio.new_event_loop()
        asyncio.set_event_loop(self.loop)  # Set the event loop for this thread.
        self.client = BleakClient(address, loop=self.loop)  # Ensure BleakClient uses this loop.
        self.thread = threading.Thread(target=self.loop.run_forever)
        self.thread.daemon = True
        self.connected = False

    def run_loop(self, loop):
        """
        Runs the given event loop.

        Parameters:
            loop (asyncio.new_event_loop): The event loop to run.
        """
        asyncio.set_event_loop(loop)
        loop.run_forever()

    async def connect(self):
        """
        Asynchronously connects to the BLE device.

        Returns:
            bool: True if the connection was successful, False otherwise.
        """
        self.connected = await self.client.connect()
        return self.connected

    async def disconnect(self):
        """
        Asynchronously disconnects from the BLE device.
        """
        await self.client.disconnect()
        self.connected = False

    async def read_characteristic(self, char_uuid):
        """
        Reads data from a specified characteristic.

        Parameters:
            char_uuid (str): The UUID of the characteristic to read from.

        Returns:
            The data read from the characteristic.
        """
        return await self.client.read_gatt_char(char_uuid)

    async def write_characteristic(self, char_uuid, data):
        """
        Writes data to a specified characteristic.

        Parameters:
            char_uuid (str): The UUID of the characteristic to write to.
            data: The data to write.
        """
        await self.client.write_gatt_char(char_uuid, data)

    def start(self):
        """
        Starts the thread and the event loop.
        """
        self.thread.start()

    def stop(self):
        """
        Stops the event loop and joins the thread.
        """
        self.loop.call_soon_threadsafe(self.loop.stop)
        self.thread.join()

    def perform_action(self, coro):
        """
        Schedules a coroutine to be run in the event loop thread.

        Parameters:
            coro: The coroutine to schedule.

        Returns:
            The result of the coroutine, or None if an error occurred.
        """
        fut = asyncio.run_coroutine_threadsafe(coro, self.loop)
        try:
            result = fut.result()
            return result
        except Exception as e:
            print(f"An error occurred: {e}")
            return None

    def prot_open(self):
        """
        Placeholder for opening protocol. Not implemented yet.
        """
        raise NotImplementedError("prot_open function not implemented yet")

    def prot_close(self):
        """
        Placeholder for closing protocol. Not implemented yet.
        """
        raise NotImplementedError("prot_close function not implemented yet")

    def prot_stop(self):
        """
        Placeholder for stopping protocol. Not implemented yet.
        """
        raise NotImplementedError("prot_stop function not implemented yet")




class FestClientThread(BleakClientThread):
    """
    A class to manage specific BLE operations for Indy devices using the BleakClientThread as a base.

    Attributes:
        OPEN_MOTOR_HDLE (str): UUID for opening the motor characteristic.
        CLOSE_MOTOR_HDLE (str): UUID for closing the motor characteristic.
        data_go (bytearray): Data to send to initiate an action.
        data_nogo (bytearray): Data to send to stop an action.
    """
    def __init__(self, address):
        """
        Initializes the IndyClientThread with a given BLE device address.

        Parameters:
            address (str): The BLE device address.
        """
        super().__init__(address)
        self.OPEN_MOTOR_HDLE = "43680002-4d74-1001-726b-526f64696f6e"
        self.CLOSE_MOTOR_HDLE = "43680003-4d74-1001-726b-526f64696f6e"
        self.data_go = bytearray([0x01])
        self.data_nogo = bytearray([0x00])

        self.MOVE_ALL_FINGERS_NEW_VM = "4368000a-4d74-1001-726b-526f64696f6e" #  6 байт по положению на каждый палец
        self.set_position = bytearray([0x00, 0x00, 0x00, 0x00, 0x00, 0x00]) # 6 байт по положению на каждый палец

        self.MIO_MEASUREMENT_NEW_VM = "43680201-4d74-1001-726b-526f64696f6e" # первые 2 байта сигналы датчиков, 3-й байт активный в данное время жест, 4-9 байты текущие углы энкодеров пальцев, 10-й байт код ошибки(для отладки, расшифровку кодов можно спросить у Лёши), 11-12 байты сокращённый UUID последней характеристики запрошенной с телефона (для системы подтверждения приёма)
        
        
    def prot_close(self):
        """
        Initiates the protocol to close the motor by writing to the CLOSE_MOTOR_HDLE characteristic.
        """
        self.perform_action(self.write_characteristic(self.CLOSE_MOTOR_HDLE, self.data_go))

    def prot_open(self):
        """
        Initiates the protocol to open the motor by writing to the OPEN_MOTOR_HDLE characteristic.
        """
        self.perform_action(self.write_characteristic(self.OPEN_MOTOR_HDLE, self.data_go))

    def prot_stop(self):
        """
        Stops any ongoing motor action by writing to both the OPEN_MOTOR_HDLE and CLOSE_MOTOR_HDLE characteristics.
        """
        self.perform_action(self.write_characteristic(self.CLOSE_MOTOR_HDLE, self.data_nogo))
        self.perform_action(self.write_characteristic(self.OPEN_MOTOR_HDLE, self.data_nogo))


In [2]:
from MotoricaInterface.utils.helpers import ble_find_fest_address

fest_devices = await ble_find_fest_address()
if fest_devices:
    for uuid, device in fest_devices:
        print(f"FEST device found: {device.name} at UUID: {uuid}")
else:
    print("No FEST devices found.")



FEST device found: FEST-XFTHS03607 at UUID: 4554C4E9-A937-3EF7-78B5-BDF2B57C243C


In [3]:
print(f'using {fest_devices[0][1].name}... ', end='')
fest_client = FestClientThread(fest_devices[0][1].address)
fest_client.start()

connected = fest_client.perform_action(fest_client.connect())
assert(connected == True)
assert(fest_client.connected == True)


print('connected!')

using FEST-XFTHS03607... connected!


In [4]:
fest_client.prot_open()
time.sleep(1)
fest_client.prot_close()
time.sleep(1)
fest_client.prot_stop()



In [5]:
import numpy as np

In [6]:
async def notification_handler(sender, data):
    # Daten in numpy-Array umwandeln
    # values = np.frombuffer(data, dtype=np.uint32)
    print(data, end = '\r')

In [19]:
async def f():
    notifications = []
    for _ in range(10):
        notification = await fest_client.client.read_gatt_char('43680202-4d74-1001-726b-526f64696f6e')
        notifications.append(notification)
        await asyncio.sleep(0.1)
    return notification

ans = await f()


RuntimeError: Task <Task pending name='Task-19' coro=<InteractiveShell.run_cell_async() running at /usr/local/anaconda3/envs/opto3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3311> cb=[IPythonKernel._cancel_on_sigint.<locals>.cancel_unless_done(<Future pendi...ernel.py:315]>)() at /usr/local/anaconda3/envs/opto3/lib/python3.11/site-packages/ipykernel/ipkernel.py:315, Task.task_wakeup()]> got Future <Future pending> attached to a different loop

In [15]:
ans

<frozen genericpath>:77: RuntimeWarning: coroutine 'f' was never awaited


TypeError: 'coroutine' object is not callable

In [ ]:
characteristics = fest_client.client.services.characteristics
for uuid, characteristic in characteristics.items():
    print(f"Characteristic UUID: {characteristic.uuid}, Properties: {characteristic.properties}")


Characteristic UUID: 43680201-4d74-1001-726b-526f64696f6e, Properties: ['notify']
Characteristic UUID: 43680200-4d74-1001-726b-526f64696f6e, Properties: ['read', 'write']
Characteristic UUID: 43680202-4d74-1001-726b-526f64696f6e, Properties: ['read']
Characteristic UUID: 43680000-4d74-1001-726b-526f64696f6e, Properties: ['read', 'write']
Characteristic UUID: 43680001-4d74-1001-726b-526f64696f6e, Properties: ['read', 'write']
Characteristic UUID: 43680004-4d74-1001-726b-526f64696f6e, Properties: ['read', 'write']
Characteristic UUID: 43680005-4d74-1001-726b-526f64696f6e, Properties: ['read', 'write']
Characteristic UUID: 43680006-4d74-1001-726b-526f64696f6e, Properties: ['read', 'write']
Characteristic UUID: 43680007-4d74-1001-726b-526f64696f6e, Properties: ['read', 'write']
Characteristic UUID: 43680008-4d74-1001-726b-526f64696f6e, Properties: ['read', 'write']
Characteristic UUID: 4368000c-4d74-1001-726b-526f64696f6e, Properties: ['read', 'write']
Characteristic UUID: 43680203-4d74-10